# 🧱 Celda 1: Carga de Datos y Codificación Numérica

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Cargamos el dataset consolidado
df = pd.read_csv('../data/processed/df_consolidado_clean.csv')

# Eliminamos columnas de identificación o constantes que queden en el dataset
columnas_omitir = ['EmployeeID', 'EmployeeCount', 'Over18', 'StandardHours']
df_modelo = df.drop(columns=[col for col in columnas_omitir if col in df.columns]).copy()

# 💡 INGENIERÍA DE VARIABLES UTILIZANDO EL NOMBRE CORRECTO: 'MonthlyIncome'
df_modelo['Sueldo_Por_Hora'] = df_modelo['MonthlyIncome'] / (df_modelo['Media_Horas_Diarias'] + 1)

# 2. Convertimos las variables categóricas (texto) a numéricas binarias (0 y 1)
df_procesado = pd.get_dummies(df_modelo, drop_first=True)

print("--- PREPROCESAMIENTO E INGENIERÍA COMPLETADOS ---")
print(f"Dataset listo para la IA: {df_procesado.shape[0]} filas y {df_procesado.shape[1]} columnas.")

--- PREPROCESAMIENTO E INGENIERÍA COMPLETADOS ---
Dataset listo para la IA: 4410 filas y 42 columnas.


# 🧱 Celda 2: División de Datos (Train/Test Split) y Escalado.

In [44]:
# 1. Separamos la variable objetivo (y) de las predictoras (X)
X = df_procesado.drop(columns=['Dias_Absentismo'])
y = df_procesado['Dias_Absentismo']

# 2. Dividimos en Entrenamiento (80%) y Prueba (20%) con una semilla fija (random_state) para que siempre dé igual
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Escalamos las variables numéricas para que tengan el mismo peso matemático (Media 0, Desviación 1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- DIVISIÓN DE DATOS COMPLETADA ---")
print(f"Empleados para entrenar la IA (X_train): {X_train_scaled.shape[0]}")
print(f"Empleados para evaluar la IA (X_test): {X_test_scaled.shape[0]}")

--- DIVISIÓN DE DATOS COMPLETADA ---
Empleados para entrenar la IA (X_train): 3528
Empleados para evaluar la IA (X_test): 882


# 🧱 Celda 3: Modelos de Regresión de Riesgo de Absentismo

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Transformamos el problema a CLASIFICACIÓN
y_train_class = (y_train >= 25).astype(int)
y_test_class = (y_test >= 25).astype(int)

# 2. Modelos Avanzados con penalización de peso (class_weight='balanced')
modelo_log_opt = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
modelo_forest_opt = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42)

# 3. Entrenamos usando los datos escalados
modelo_log_opt.fit(X_train_scaled, y_train_class)
modelo_forest_opt.fit(X_train_scaled, y_train_class)

# 4. Predecimos
pred_log_opt = modelo_log_opt.predict(X_test_scaled)
pred_forest_opt = modelo_forest_opt.predict(X_test_scaled)

print("--- MODELOS OPTIMIZADOS CON INGENIERÍA DE VARIABLES ---")
print(f"Precisión Final Regresión Logística: {accuracy_score(y_test_class, pred_log_opt):.2%}")
print(f"Precisión Final Random Forest: {accuracy_score(y_test_class, pred_forest_opt):.2%}")

--- MODELOS OPTIMIZADOS CON INGENIERÍA DE VARIABLES ---
Precisión Final Regresión Logística: 54.65%
Precisión Final Random Forest: 53.63%


# 🧱 Celda 4: Evaluación Detallada y Reporte de Métricas

In [46]:
from sklearn.metrics import classification_report, confusion_matrix

# Generamos el reporte detallado para nuestro modelo base de clasificación
reporte = classification_report(y_test_class, pred_log_opt, target_names=['Riesgo Normal (<25 días)', 'Riesgo Alto (>=25 días)'])

print("--- REPORTE DE CLASIFICACIÓN DETALLADO ---")
print(reporte)

print("\n--- MATRIZ DE CONFUSIÓN ---")
matriz = confusion_matrix(y_test_class, pred_log_opt)
print(f"Verdaderos Normales detectados: {matriz[0][0]}")
print(f"Falsos Altos (Errores): {matriz[0][1]}")
print(f"Falsos Normales (Errores): {matriz[1][0]}")
print(f"Verdaderos Altos detectados: {matriz[1][1]}")

--- REPORTE DE CLASIFICACIÓN DETALLADO ---
                          precision    recall  f1-score   support

Riesgo Normal (<25 días)       0.54      0.50      0.52       434
 Riesgo Alto (>=25 días)       0.55      0.59      0.57       448

                accuracy                           0.55       882
               macro avg       0.55      0.55      0.55       882
            weighted avg       0.55      0.55      0.55       882


--- MATRIZ DE CONFUSIÓN ---
Verdaderos Normales detectados: 217
Falsos Altos (Errores): 217
Falsos Normales (Errores): 183
Verdaderos Altos detectados: 265


## 💾 4. Guardado del Modelo Entrenado y Escalador

In [47]:
import joblib

# 1. Guardamos el modelo optimizado con su nombre correcto
joblib.dump(modelo_log_opt, 'modelo_absentismo.pkl')

# 2. Guardamos el escalador que dejamos listo en la Celda 2
joblib.dump(scaler, 'escalador_absentismo.pkl')

print("¡Modelo y Escalador guardados con éxito en archivos .pkl!")

¡Modelo y Escalador guardados con éxito en archivos .pkl!
